# Sim sim salabim

## Task 1. Energy Drift in Numerical Integration (5 pts)

In this task you will explore how the choice of ODE integration method affects energy conservation in a physics simulation.

We simulate a free rigid body (a box) with initial angular momentum $\mathbf{L}_0 = (10, 10, 0)$ and no external forces or gravity. Since no work is done on the system, the total kinetic energy must remain **constant** over time. However, numerical integration methods introduce errors at each time step, and these errors can cause the computed energy to **drift** — typically growing without bound for the forward Euler method.

#### Note: Callbacks in `PhysicsEngine`

Since the seminar, `PhysicsEngine` has been extended with a `callbacks` field — a list of callable objects that are invoked **after every simulation step** (i.e. after the new state has been computed and applied to the bodies). Each callback receives the `PhysicsEngine` instance as its only argument, giving it access to the current `time` and the list of `bodies`. See `PhysicsEngine.step()` for details:

```python
for callback in self.callbacks:
    callback(self)
```

The `KinEnergyCallback` class in `solutions/kin_energy.py` is an example of such a callback: it computes kinetic energy after each step, records it in `kin_e_history`, and plots it on a live graph.

### 1.1. Kinetic Energy Callback

**File:** `solutions/kin_energy.py` — method `KinEnergyCallback.calc_kinetic_energy`

Implement a method that computes the **total kinetic energy** of the system. For each rigid body, the kinetic energy consists of two parts:

$$E_{\text{kin}} = \sum_i \left( \underbrace{\frac{1}{2}\, m_i \, \mathbf{v}_i \cdot \mathbf{v}_i}_{\text{translational}} \;+\; \underbrace{\frac{1}{2}\, \boldsymbol{\omega}_i \cdot \mathbf{L}_i}_{\text{rotational}} \right)$$

where:
- $m_i = 1 / \texttt{body.inv\_mass}$ — mass of the body,
- $\mathbf{v}_i$ = `body.body_velocity` — linear velocity,
- $\boldsymbol{\omega}_i$ = `body.angular_velocity` — angular velocity,
- $\mathbf{L}_i$ = `body.angular_momentum` — angular momentum.

> **Note:** Skip bodies with zero mass ($\texttt{inv\_mass} = 0$, i.e. static bodies).

After implementing, run `python scripts/kin_energy.py --solver euler` and observe the energy plot. You should see the kinetic energy **growing over time** — this is the numerical energy drift caused by the forward Euler method.

### 1.2 Runge-Kutta Integration

**File:** `solutions/ode_solvers.py` — method `RK4Method.ode`

The forward Euler method is only 1st-order accurate, and its energy drift makes it unsuitable for long-running simulations. A much better alternative is the **classic 4th-order Runge-Kutta (RK4)** method.

Given a system $\dot{\mathbf{y}} = f(\mathbf{y})$, one RK4 step from $\mathbf{y}_n$ with step size $\Delta t$ is:

$$\mathbf{k}_1 = f(\mathbf{y}_n)$$
$$\mathbf{k}_2 = f\!\left(\mathbf{y}_n + \tfrac{\Delta t}{2}\,\mathbf{k}_1\right)$$
$$\mathbf{k}_3 = f\!\left(\mathbf{y}_n + \tfrac{\Delta t}{2}\,\mathbf{k}_2\right)$$
$$\mathbf{k}_4 = f\!\left(\mathbf{y}_n + \Delta t\,\mathbf{k}_3\right)$$
$$\mathbf{y}_{n+1} = \mathbf{y}_n + \frac{\Delta t}{6}\left(\mathbf{k}_1 + 2\mathbf{k}_2 + 2\mathbf{k}_3 + \mathbf{k}_4\right)$$

#### Implementation details

In our simulation, the state is represented as a dictionary with two keys:
- `'x'` — positions / orientations (generalized coordinates),
- `'p'` — linear and angular momenta (generalized momenta).

The derivative function `dx_dt_func(state)` returns a dictionary with:
- `'x_dot'` — time derivatives of coordinates,
- `'p_dot'` — time derivatives of momenta (i.e. forces / torques).

You can use the `EulerMethod` implementation as a reference for how to call `dx_dt_func` and construct the result.

#### Verification

After implementing RK4, run:

```bash
python scripts/kin_energy.py --solver rk4
```

Compare the energy plot with the Euler result. With RK4, the kinetic energy should remain **nearly constant** — demonstrating that a higher-order integrator dramatically reduces energy drift.

You can verify your implementation by running the tests:

```bash
pytest tests/test_kin_energy.py -v
```

## Task 2. Joints (5 pts)
### 2.1 Updated Constraint Manager 

**File:** `solutions/constraints_manager.py` — method `ConstraintsManagerImpulseBased.calc_forces`

In the seminar we derived constraint forces by solving for the Lagrange multipliers $\lambda$ at the **acceleration level**:

$$JWJ^T \lambda = -\dot{J}V - JWF - k_d JV - k_s C$$

Here $F_C = J^T \lambda$ are the constraint forces, and the terms $-k_d JV - k_s C$ are Baumgarte stabilisation corrections added in acceleration space.

An equally valid alternative is to work at the **velocity level**. The idea is to pre-integrate the unconstrained velocity for the next timestep and then find an **impulse** $\lambda$ that corrects it:

1. Compute the velocity that external forces alone would produce:
$$V_1 = V_0 + W F \,\Delta t$$

2. Solve for $\lambda$ such that after applying the impulse $J^T\lambda$ the resulting velocity satisfies the constraint, with a small bias term that corrects position-level drift:
$$JWJ^T \lambda = -J V_1 - \frac{\beta}{\Delta t} C$$

3. Convert the impulse back to a force:
$$F_C = \frac{J^T \lambda}{\Delta t}$$

The bias velocity $\frac{\beta}{\Delta t} C$ plays the same stabilising role as the $k_s C$ term in the acceleration-level formulation, but expressed directly as a target velocity correction. The single parameter $\beta \in [0, 1]$ controls how aggressively constraint drift is corrected each timestep.

A practical advantage of this formulation is that **it does not require implementing `get_J_dot_V`** for every constraint — the effect of $\dot{J}V$ is implicitly absorbed into $V_1$.

**Your task:** implement `ConstraintsManagerImpulseBased.calc_forces` in `solutions/constraints_manager.py`. The code above the `# YOUR CODE HERE` marker shows the setup; fill in the missing computation.

### 2.2 Ball-and-Socket Joint

**File:** `solutions/constraints.py` — class `BallAndSocketJoint`

A Ball-and-Socket joint connects two rigid bodies at a shared anchor point, allowing arbitrary rotation between them but preventing any relative translation. It removes 3 translational degrees of freedom and has the following constraint equation:

$$C = \mathbf{r}_2 + \mathbf{r}_{f_2} - \mathbf{r}_1 - \mathbf{r}_{f_1} = 0$$

where $\mathbf{r}_1$, $\mathbf{r}_2$ are the centres of mass of the two bodies, and $\mathbf{r}_{f_1}$, $\mathbf{r}_{f_2}$ are the anchor offsets from each COM expressed in the **world frame** (updated every timestep by rotating the local anchor vectors by the current body orientations).

#### Jacobian

The generalised velocity of the system is $V = (\dot{\mathbf{r}}_1,\, \boldsymbol{\omega}_1,\, \dot{\mathbf{r}}_2,\, \boldsymbol{\omega}_2)^T$. Differentiating $C$ with respect to time:

$$\dot{C} = \dot{\mathbf{r}}_2 + \boldsymbol{\omega}_2 \times \mathbf{r}_{f_2} - \dot{\mathbf{r}}_1 - \boldsymbol{\omega}_1 \times \mathbf{r}_{f_1} = J\,V$$

From $\dot{C} = JV$ you can read off the Jacobian blocks $J_1$ and $J_2$ for each body. You may find the skew-symmetric matrix identity $\boldsymbol{\omega} \times \mathbf{r}_f = -[\mathbf{r}_f]_\times\, \boldsymbol{\omega}$ useful.

#### Your task

Implement the following two methods in `BallAndSocketJoint`:

- **`get_C(self)`** — return the 3-vector $C = \mathbf{r}_2 + \mathbf{r}_{f_2} - \mathbf{r}_1 - \mathbf{r}_{f_1}$.
- **`get_J_updates(self)`** — return a dictionary `{body_1.name: J_1, body_2.name: J_2}` with the $3 \times 6$ Jacobian blocks derived above.

> **Note:** `get_J_dot_V` is not required here — the `ConstraintsManagerImpulseBased` formulation from task 2.1 absorbs this term implicitly.

You can visualise the result by running:

```bash
python scripts/bs_joint.py
```

You should see two rods connected at their ends, swinging under gravity.

You can verify your implementation by running the tests:

```bash
pytest tests/test_joints.py -v
```

### 2.3 Drift in Constraint Forces *(optional, not tested)*

This task is exploratory — no code submission is required.

#### Observing the drift

Add a `KinEnergyCallback` to the simulation in `scripts/bs_joint.py` and **disable gravity** (remove or comment out the `GravityForce` entry in `forces`). With no external forces acting, the total kinetic energy of the system should be conserved. Run the simulation and observe the energy plot — you will see the energy **growing over time**, similar to what you saw in Task 1.1 with the forward Euler integrator.

#### Why does energy drift with RK4?

In Task 1.2 you saw that RK4 keeps energy nearly constant for a free rigid body. So why does drift reappear here?

The reason is that **constraint forces are computed once per timestep and then held constant** throughout the RK4 sub-steps. RK4 calls the derivative function $f(y)$ four times at intermediate states $y_n,\, y_n + \frac{\Delta t}{2}k_1,\, \ldots$, but each of these calls uses the same $F_C$ that was computed at the beginning of the step. The constraint forces therefore lag behind the actual state of the system, introducing an inconsistency that causes energy to leak.

In contrast, for the free rigid body in Task 1 there were no constraint forces at all, so RK4 had nothing extra to get wrong.

#### A possible fix

A proper solution is to make constraint forces part of the derivative function passed to the ODE solver, so that RK4 can re-evaluate them at each intermediate state:

1. At each call to `system_dx_dt_func(state)` inside `PhysicsEngine`, temporarily set the body states to `state`, recompute the constraint forces, add them to the force accumulators, and then evaluate the derivatives as usual.
2. This way RK4 sees a consistent $(F_{\text{ext}} + F_C)$ at every sub-step, restoring the accuracy of the integrator.

This integration is non-trivial to implement cleanly (constraint forces depend on body state, which changes at each RK4 stage), and is left as an open-ended exercise for the curious.

## Task 3. Contact Forces. Penalty Method (5 pts)

In this task you have to implement normal and friction force computation in `solutions/penalty.py` for each collision event.

### 3.1 Collision detection

Collision detection is handled by the [python-fcl](https://github.com/BerkeleyAutomation/python-fcl) library. Install it with:

```bash
pip install python-fcl
```

Each detected collision is represented by a `Collision` object (defined in `lib/phys/collisions/collision_detector.py`) with the following fields:

| Field | Type | Description |
|---|---|---|
| `obj_a` | `RigidBody \| Particle` | First colliding body |
| `obj_b` | `RigidBody \| Particle` | Second colliding body (may be static, i.e. `inv_mass = 0`) |
| `point_a` | `Vec3` | Contact point relative to the COM of `obj_a` |
| `point_b` | `Vec3` | Contact point relative to the COM of `obj_b` |
| `normal` | `Vec3` | Contact normal, pointing **from** `obj_b` **into** `obj_a` |
| `depth` | `float` | Penetration depth (positive when objects overlap) |

 

> **Note:** When two bodies collide, there are two collision events (`obj_a` vs `obj_b` and `obj_b` vs `obj_a`).

### 3.2 Normal Force 

**File:** `solutions/penalty.py` — method `PenaltyMethod.step`

#### Penalty method

The penalty method models a contact as a **spring-damper** acting along the contact normal. When two objects overlap by depth $d$, a restoring force is applied proportional to the penetration and the relative normal velocity:

$$F_n = -(k_s \, d + k_d \, v_n)$$

where:
- $k_s$ — spring stiffness (`self.k_s`),
- $k_d$ — damping coefficient (`self.k_d`),
- $d$ = `collision.depth` — penetration depth,
- $v_n = \mathbf{v}_{\mathrm{rel}} \cdot \hat{n}$ — relative velocity projected onto the contact normal.

The relative velocity at the contact point must account for the angular velocity of rigid bodies:

$$\mathbf{v}_{\mathrm{contact}} = \mathbf{v}_{\mathrm{COM}} + \boldsymbol{\omega} \times \mathbf{r}$$

where $\mathbf{r}$ is the vector from the COM to the contact point (`point_a` or `point_b`).

The force $\mathbf{F}_n = F_n\, \hat{n}$ is applied to `obj_a`. If `obj_a` is a rigid body, also apply the corresponding torque $\boldsymbol{\tau} = \mathbf{r}_a \times \mathbf{F}_n$.

> **Note:** apply forces only to `obj_a`.

### 3.3 Friction

**File:** `solutions/penalty.py` — method `PenaltyMethod.step` (**same block, after the normal force**)

Once the normal force is computed, add a tangential **friction force** that opposes the relative motion at the contact point.

Use the following model:

1. Compute a viscous friction force from the full relative velocity at the contact point:
$$\mathbf{f} = -k_{\mathrm{drag}}\, \mathbf{v}_{\mathrm{rel}}$$

2. Clamp it to the **Coulomb friction limit** to prevent the friction force from exceeding the normal force scaled by the friction coefficient $\mu$:
$$\mathbf{f} \leftarrow \begin{cases} \mathbf{f} & \text{if } |\mathbf{f}| \leq \mu\,|\mathbf{F}_n| \\ \mu\,|\mathbf{F}_n|\, \dfrac{\mathbf{f}}{|\mathbf{f}|} & \text{otherwise} \end{cases}$$

where:
- $k_{\mathrm{drag}}$ = `self.k_drag` — viscous drag coefficient,
- $\mu$ = `self.mu` — Coulomb friction coefficient,
- $\mathbf{F}_n$ — normal force computed in task 3.1.

Apply $\mathbf{f}$ to `obj_a` as a force and, if `obj_a` is a rigid body, as a torque $\boldsymbol{\tau} = \mathbf{r}_a \times \mathbf{f}$.

You can visualise the result by running:

```bash
python scripts/penalty.py
```

You should see two boxes bouncing inside a closed room under gravity.

You can verify your implementation by running the tests:

```bash
pytest tests/test_penalty.py -v
```